# 05 - Assistente medico com LangChain (RAG + prontuario)

Demonstra o pipeline do assistente (`src/assistant/chain.py`): consulta ao "prontuario" (SQLite sobre o dataset de stroke), busca semantica (RAG/FAISS) nos protocolos/FAQs, geracao da resposta e aplicacao do guardrail de seguranca. Usa o backend LLM local (fine-tuned) se o adapter da Fase 04 estiver disponivel, caso contrario cai para o Gemini (`src/llm/client.py`), da Fase 2.

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/stroke-prediction')
    sys.path.insert(0, str(ROOT))
    !pip install -r "{ROOT / 'requirements.txt'}"
    print()
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    sys.path.insert(0, str(ROOT))
    print("Rodando Localmente")

from src.assistant.chain import answer_question
from src.assistant.llm_backend import get_generate_fn, local_adapter_available
from src.assistant.patient_db import build_patient_db
from src.assistant.retriever import build_vectorstore

print('Adapter fine-tuned local disponivel?', local_adapter_available())

In [ ]:
build_patient_db()
vectorstore = build_vectorstore()
generate_fn = get_generate_fn()
print('Assistente pronto.')

## Pergunta 1 — paciente de alto risco, sobre criterios de trombolise

In [ ]:
result = answer_question(patient_id=9046, question='Quais os criterios para considerar trombolise neste paciente?', vectorstore=vectorstore, generate_fn=generate_fn)
print('--- Resposta ---')
print(result['response'])
print()
print('Fontes citadas:', result['sources'])
print('Exames pendentes:', result['pending_exams'])
print('Requer validacao humana?', result['requires_human_validation'])

## Pergunta 2 — orientacao a familiares (escala FAST)

In [ ]:
result2 = answer_question(patient_id=51676, question='Como explico a escala FAST para a familia deste paciente?', vectorstore=vectorstore, generate_fn=generate_fn)
print(result2['response'])
print()
print('Fontes citadas:', result2['sources'])

## Explainability

Cada resposta lista as fontes (`sources`) recuperadas pelo RAG que embasaram o texto gerado — atende ao requisito de explainability do desafio (indicar a origem da informacao usada).